# Exercise 006

<a href="https://colab.research.google.com/github/FAIRChemistry/PythonProgramming2025/blob/master/exercises/Exercise006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Please execute this cell to download the necessary data
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2026/master/data/all_sequences.fasta

--2026-06-09 13:54:25--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2026/master/data/all_sequences.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2940315 (2.8M) [text/plain]
Saving to: ‘all_sequences.fasta’

all_sequences.fasta 100%[===================>]   2.80M  --.-KB/s    in 0.02s   

2026-06-09 13:54:25 (185 MB/s) - ‘all_sequences.fasta’ saved [2940315/2940315]



# DNASequence class

Read the FASTA file `all_sequences.fasta` and store header info and sequence in a suitable class. Make sure that at the initialization of the object, the following atrributes are present:

* `id`
* `organism`
* `sequence`
* `gc_content`
* `length`

**Tips**

> * Your `__init__`-method arguments do not have to contain all expected attributes if you can derive them from another attribute. The `__init__`-method is a function and you can execute any code you want upon initialization. Make sure to assign your calculation to the appropriate attribute via `self.xyz`.
> * [Dataclasses](https://docs.python.org/3/library/dataclasses.html) are a convinient way to create classes that simply hold data. You can make use of them to simplify the process due to the automatic generation of a `__init__`-method. But keep in mind that this excludes additional calculation you would have otherwise put into your custom `__init__`-method.

In [2]:
class DNASequence:
    def __init__(self, header: str, sequence: str):
        # Clean the header and split it to extract ID and Organism
        # We assume a standard format where the first word is the ID
        header_clean = header.strip().lstrip('>')
        parts = header_clean.split(' ', 1)

        self.id = parts[0]
        # If there's no organism provided after the ID, default to "Unknown"
        self.organism = parts[1] if len(parts) > 1 else "Unknown"

        # Store sequence and calculate length
        self.sequence = sequence.upper()
        self.length = len(self.sequence)

        # Calculate GC content
        if self.length > 0:
            gc_count = self.sequence.count('G') + self.sequence.count('C')
            self.gc_content = gc_count / self.length
        else:
            self.gc_content = 0.0

    def __repr__(self):
        # A helpful representation for when you print the object
        return f"<DNASequence(id={self.id}, gc_content={self.gc_content:.2%}, length={self.length})>"

# --- Parsing the FASTA file ---
def read_fasta(file_path):
    dna_objects = []
    with open(file_path, 'r') as file:
        header = ""
        seq_lines = []
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    # Initialize our custom object and add to list
                    dna_objects.append(DNASequence(header, ''.join(seq_lines)))
                header = line
                seq_lines = []
            else:
                seq_lines.append(line)

## Magic Methods - Alignment by `==`

Can you extend the class to output the identity between the two sequences (stored as an attribute) when the `==` comparison operator is used? Apply the implementation to two sequences that you have chosen and use the supplied `get_identity` function.

Learn more about [Magic methods](https://realpython.com/python-magic-methods/)

In [3]:
# Execute this cell to install all necessary packages
%pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.5 MB/s eta 0:00:00


In [4]:
# Execute this cell to use the alignment function
from Bio import pairwise2


def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython

    Args:
        seq1 (str): Query sequence to align to
        seq2 (str): Target sequence to align with

    Returns:
        float: Identity of the resulting alignment

    """
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [6]:
from Bio import pairwise2

# 1. The original class and parser
class DNASequence:
    def __init__(self, header: str, sequence: str):
        header_clean = header.strip().lstrip('>')
        parts = header_clean.split(' ', 1)
        self.id = parts[0]
        self.organism = parts[1] if len(parts) > 1 else "Unknown"
        self.sequence = sequence.upper()
        self.length = len(self.sequence)

        if self.length > 0:
            gc_count = self.sequence.count('G') + self.sequence.count('C')
            self.gc_content = gc_count / self.length
        else:
            self.gc_content = 0.0

def read_fasta(file_path):
    dna_objects = []
    with open(file_path, 'r') as file:
        header = ""
        seq_lines = []
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    dna_objects.append(DNASequence(header, ''.join(seq_lines)))
                header = line
                seq_lines = []
            else:
                seq_lines.append(line)
        if header:
            dna_objects.append(DNASequence(header, ''.join(seq_lines)))
    return dna_objects

# 2. Parse the file to define 'sequences' right now
sequences = read_fasta("all_sequences.fasta")

# 3. The alignment function
def get_identity(seq1: str, seq2: str):
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)

# 4. The extended class
class DNASequenceWithAlignment(DNASequence):
    def __eq__(self, other):
        if isinstance(other, DNASequence):
            return get_identity(self.sequence, other.sequence)
        return NotImplemented

# 5. The execution
if len(sequences) >= 2:
    seq_obj1 = DNASequenceWithAlignment(f">{sequences[0].id} {sequences[0].organism}", sequences[0].sequence)
    seq_obj2 = DNASequenceWithAlignment(f">{sequences[1].id} {sequences[1].organism}", sequences[1].sequence)

    alignment_score = seq_obj1 == seq_obj2

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs

# --- 1. The Helper Functions ---
def initialize_centroids(X, n_clusters):
    np.random.seed(42)
    random_indices = np.random.permutation(X.shape[0])
    return X[random_indices[:n_clusters]]

def assign_labels(X, centroids):
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
    return np.argmin(distances, axis=1)

def compute_centroids(X, labels, n_clusters):
    centroids = np.zeros((n_clusters, X.shape[1]))
    for k in range(n_clusters):
        cluster_points = X[labels == k]
        if len(cluster_points) > 0:
            centroids[k] = np.mean(cluster_points, axis=0)
    return centroids

def has_converged(old_centroids, new_centroids, tol):
    return np.all(np.abs(old_centroids - new_centroids) < tol)

# --- 2. The Custom KMeans Implementation ---
def kmeans(df, n_clusters=3, max_iter=300, tol=1e-4):
    # Convert dataframe to numpy array
    X_df = df.select_dtypes(include="number")
    X = X_df.to_numpy()

    # Algorithm loop
    centroids = initialize_centroids(X, n_clusters)
    labels = np.zeros(X.shape[0])

    for iteration in range(max_iter):
        labels = assign_labels(X, centroids)
        new_centroids = compute_centroids(X, labels, n_clusters)

        if has_converged(centroids, new_centroids, tol):
            centroids = new_centroids
            break

        centroids = new_centroids

    return centroids, labels

# --- 3. The Verification Script ---
# Generate fake data
X, true_labels = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)
test_df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])

# Run custom algorithm
centroids, predicted_labels = kmeans(test_df, n_clusters=3, max_iter=300, tol=1e-4)
test_df['Predicted_Cluster'] = predicted_labels